# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chisman001/ML-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [51]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [52]:
import duckdb

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

In [53]:
con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    );
""")

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [54]:
march_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

con.sql(f"""
    SELECT COUNT(*) AS total_rows
    FROM read_parquet('{march_path}')
""").show()

┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│    9841378 │
└────────────┘



In [55]:
content_path = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

con.sql(f"""
    SELECT
        COUNT(*) AS total_content,
        MIN(content_created_date) AS earliest_created,
        MAX(content_created_date) AS latest_created,
        MIN(content_updated_date) AS earliest_updated,
        MAX(content_updated_date) AS latest_updated
    FROM read_parquet('{content_path}')
""").show()

┌───────────────┬──────────────────┬────────────────┬──────────────────┬────────────────┐
│ total_content │ earliest_created │ latest_created │ earliest_updated │ latest_updated │
│     int64     │       date       │      date      │       date       │      date      │
├───────────────┼──────────────────┼────────────────┼──────────────────┼────────────────┤
│        519606 │ 2024-10-16       │ 2026-07-06     │ 2024-10-28       │ 2026-07-06     │
└───────────────┴──────────────────┴────────────────┴──────────────────┴────────────────┘



In [56]:
con.sql(f"""
    SELECT
        CASE
            WHEN date_diff('day', c.content_updated_date, DATE '2026-03-31') < 90
                THEN '<3 months'
            WHEN date_diff('day', c.content_updated_date, DATE '2026-03-31') < 180
                THEN '3-6 months'
            WHEN date_diff('day', c.content_updated_date, DATE '2026-03-31') < 365
                THEN '6-12 months'
            ELSE '12+ months'
        END AS content_age_bucket,

        COUNT(*) AS n,

        ROUND(AVG(p.gsc_impressions), 2) AS avg_impressions,
        ROUND(AVG(p.gsc_clicks), 2) AS avg_clicks,
        ROUND(AVG(p.gsc_avg_position), 2) AS avg_position

    FROM read_parquet('{march_path}') p

    JOIN read_parquet('{content_path}') c
      ON p.client_hash_id = c.client_hash_id
     AND p.content_hash_id = c.content_hash_id

    WHERE p.gsc_data_available IS TRUE
      AND c.content_updated_date IS NOT NULL
      AND c.content_updated_date <= DATE '2026-03-31'

    GROUP BY content_age_bucket

    ORDER BY
        CASE content_age_bucket
            WHEN '<3 months' THEN 1
            WHEN '3-6 months' THEN 2
            WHEN '6-12 months' THEN 3
            WHEN '12+ months' THEN 4
        END
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬────────┬─────────────────┬────────────┬──────────────┐
│ content_age_bucket │   n    │ avg_impressions │ avg_clicks │ avg_position │
│      varchar       │ int64  │     double      │   double   │    double    │
├────────────────────┼────────┼─────────────────┼────────────┼──────────────┤
│ <3 months          │ 638964 │           53.22 │       0.12 │        14.92 │
│ 3-6 months         │   9622 │           49.51 │       0.14 │        18.86 │
│ 6-12 months        │   1617 │           10.76 │       0.02 │        17.17 │
└────────────────────┴────────┴─────────────────┴────────────┴──────────────┘



In [57]:
con.sql(f"""
    SELECT
        COUNT(*) AS joined_rows,
        COUNT(*) FILTER (
            WHERE c.content_updated_date IS NULL
        ) AS missing_update_date,
        MIN(c.content_updated_date) AS earliest_update_date,
        MAX(c.content_updated_date) AS latest_update_date
    FROM read_parquet('{march_path}') p
    JOIN read_parquet('{content_path}') c
      ON p.client_hash_id = c.client_hash_id
     AND p.content_hash_id = c.content_hash_id
    WHERE p.gsc_data_available IS TRUE
""").show()

┌─────────────┬─────────────────────┬──────────────────────┬────────────────────┐
│ joined_rows │ missing_update_date │ earliest_update_date │ latest_update_date │
│    int64    │        int64        │         date         │        date        │
├─────────────┼─────────────────────┼──────────────────────┼────────────────────┤
│     3611061 │                   0 │ 2025-06-01           │ 2026-07-06         │
└─────────────┴─────────────────────┴──────────────────────┴────────────────────┘



In [58]:
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT p.content_hash_id) AS unique_content_pages,
        MIN(c.content_updated_date) AS earliest_update_date,
        MAX(c.content_updated_date) AS latest_update_date
    FROM read_parquet('{march_path}') p
    JOIN read_parquet('{content_path}') c
      ON p.client_hash_id = c.client_hash_id
     AND p.content_hash_id = c.content_hash_id
    WHERE p.gsc_data_available IS TRUE
      AND c.content_updated_date IS NOT NULL
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────────┬──────────────────────┬────────────────────┐
│ total_rows │ unique_content_pages │ earliest_update_date │ latest_update_date │
│   int64    │        int64         │         date         │        date        │
├────────────┼──────────────────────┼──────────────────────┼────────────────────┤
│    3611061 │               176738 │ 2025-06-01           │ 2026-07-06         │
└────────────┴──────────────────────┴──────────────────────┴────────────────────┘



## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal 1 — Content staleness: MIXED

Older content shows weaker search performance in the March slice, with lower average impressions and clicks and generally worse average position. However, the older buckets contain far fewer observations than the recent-content bucket, and there are no 12+ month observations in this slice. Therefore, staleness is a useful directional signal but not strong enough to treat as a standalone decision rule.

### Signal 2 — CTR vs search position: CONFIRMED

CTR is directionally related to search position in the March slice. Pages ranking 1–3 have an aggregate CTR of 0.381%, compared with 0.131% for pages ranking 21+. This supports using CTR relative to search visibility as a signal for identifying possible content opportunities. However, position itself affects CTR, so low CTR alone should not be treated as proof that a page needs a refresh.

In [59]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position <= 3 THEN '1-3'
            WHEN gsc_avg_position <= 10 THEN '4-10'
            WHEN gsc_avg_position <= 20 THEN '11-20'
            ELSE '21+'
        END AS position_bucket,

        COUNT(*) AS n,

        ROUND(
            SUM(gsc_clicks) * 100.0 /
            NULLIF(SUM(gsc_impressions), 0),
            3
        ) AS ctr_pct,

        ROUND(AVG(gsc_impressions), 2) AS avg_impressions,
        ROUND(AVG(gsc_clicks), 2) AS avg_clicks

    FROM read_parquet('{march_path}')

    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
      AND gsc_avg_position > 0

    GROUP BY position_bucket

    ORDER BY
        CASE position_bucket
            WHEN '1-3' THEN 1
            WHEN '4-10' THEN 2
            WHEN '11-20' THEN 3
            WHEN '21+' THEN 4
        END
""").show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────┬─────────┬─────────────────┬────────────┐
│ position_bucket │    n    │ ctr_pct │ avg_impressions │ avg_clicks │
│     varchar     │  int64  │ double  │     double      │   double   │
├─────────────────┼─────────┼─────────┼─────────────────┼────────────┤
│ 1-3             │  564173 │   0.381 │           94.94 │       0.36 │
│ 4-10            │ 1456122 │   0.323 │           94.66 │       0.31 │
│ 11-20           │  519223 │   0.315 │            56.6 │       0.18 │
│ 21+             │  908354 │   0.131 │           65.41 │       0.09 │
└─────────────────┴─────────┴─────────┴─────────────────┴────────────┘



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Baseline rule:

Rank content pages using two observable signals as of March 31, 2026:
CTR opportunity and content staleness.

Pages with lower CTR than expected for their search-position range receive
higher CTR-opportunity scores. Older content receives higher staleness scores.

The final baseline score is the weighted combination of the two signals:
70% CTR opportunity and 30% staleness.

Each page receives one reason code describing the strongest signal and an
action label indicating whether it should be reviewed.

In [60]:
decision_date = "2026-03-31"

baseline_query = f"""
WITH daily AS (
    SELECT
        p.client_hash_id,
        p.content_hash_id,
        p.report_date,
        p.gsc_impressions,
        p.gsc_clicks,
        p.gsc_avg_position
    FROM read_parquet('{march_path}') p
    WHERE p.gsc_data_available IS TRUE
      AND p.gsc_impressions > 0
      AND p.gsc_avg_position > 0
),

page_month AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_avg_position) AS avg_position

    FROM daily

    GROUP BY
        client_hash_id,
        content_hash_id
),

base AS (
    SELECT
        p.client_hash_id,
        p.content_hash_id,
        p.total_impressions,
        p.total_clicks,

        CASE
            WHEN p.total_impressions > 0
            THEN p.total_clicks * 100.0 / p.total_impressions
            ELSE NULL
        END AS ctr_pct,

        p.avg_position,

        date_diff(
            'day',
            c.content_updated_date,
            DATE '{decision_date}'
        ) AS days_since_update

    FROM page_month p

    JOIN read_parquet('{content_path}') c
      ON p.client_hash_id = c.client_hash_id
     AND p.content_hash_id = c.content_hash_id

    WHERE c.content_updated_date IS NOT NULL
      AND c.content_updated_date <= DATE '{decision_date}'
),

scored AS (
    SELECT
        *,

        CASE
            -- Strong evidence: enough impressions + good position + low CTR
            WHEN total_impressions >= 100
                 AND avg_position <= 10
                 AND ctr_pct < 0.20
                THEN 70

            -- Moderate evidence
            WHEN total_impressions >= 30
                 AND avg_position <= 10
                 AND ctr_pct < 0.20
                THEN 50

            -- Some evidence
            WHEN total_impressions >= 30
                 AND avg_position <= 20
                 AND ctr_pct < 0.15
                THEN 40

            ELSE 0
        END AS ctr_score,

        CASE
            WHEN days_since_update >= 365 THEN 30
            WHEN days_since_update >= 180 THEN 20
            WHEN days_since_update >= 90 THEN 10
            ELSE 0
        END AS stale_score

    FROM base
),

final AS (
    SELECT
        *,

        ctr_score + stale_score AS score,

        CASE
            WHEN ctr_score >= 50 AND stale_score >= 20
                THEN 'LOW_CTR_AND_STALE'

            WHEN ctr_score >= 50
                THEN 'LOW_CTR'

            WHEN stale_score >= 20
                THEN 'STALE_CONTENT'

            ELSE 'LOW_PRIORITY'
        END AS reason_code,

        CASE
            WHEN ctr_score >= 50 AND stale_score >= 20
                THEN 'REVIEW_REFRESH_AND_CTR'

            WHEN ctr_score >= 50
                THEN 'REVIEW_CTR'

            WHEN stale_score >= 20
                THEN 'REVIEW_REFRESH'

            ELSE 'MONITOR'
        END AS action

    FROM scored
)

SELECT
    client_hash_id,
    content_hash_id,
    total_impressions,
    total_clicks,
    ROUND(ctr_pct, 3) AS ctr_pct,
    ROUND(avg_position, 2) AS avg_position,
    days_since_update,
    score,
    reason_code,
    action

FROM final

ORDER BY
    score DESC,
    total_impressions DESC
"""

queue = con.sql(baseline_query).df()

print("Unique content-page rows:", len(queue))
queue.head(20)

Unique content-page rows: 27831


,client_hash_id,content_hash_id,total_impressions,total_clicks,ctr_pct,avg_position,days_since_update,score,reason_code,action
0,client_c182d11e4862a37d,content_42ce26be1ec6be00,4411.0,6.0,0.136,4.26,264,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
1,client_c182d11e4862a37d,content_bea86ce3455100b0,3670.0,1.0,0.027,6.56,232,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
2,client_c182d11e4862a37d,content_5120dcbbb086843d,1429.0,0.0,0.000,6.32,247,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
3,client_c182d11e4862a37d,content_9dc017d4ef83c0d9,231.0,0.0,0.000,6.23,235,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
4,client_c182d11e4862a37d,content_e3faf06779436e84,218.0,0.0,0.000,7.97,235,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
5,client_c182d11e4862a37d,content_7d986479e843e4cd,205.0,0.0,0.000,4.13,246,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
6,client_c182d11e4862a37d,content_d91b572e9926aeea,130.0,0.0,0.000,7.18,235,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
7,client_20259bd6705d81d4,content_1d2dc429b857b82c,6376.0,8.0,0.125,4.74,124,80,LOW_CTR,REVIEW_CTR
8,client_20259bd6705d81d4,content_f8e84719d52ab1fe,3676.0,0.0,0.000,9.83,124,80,LOW_CTR,REVIEW_CTR
9,client_20259bd6705d81d4,content_96eb80b1ad849418,2134.0,4.0,0.187,6.38,124,80,LOW_CTR,REVIEW_CTR


In [61]:
queue["content_hash_id"].duplicated().sum()

np.int64(0)

In [62]:
queue["reason_code"].value_counts()

,count
reason_code,
LOW_PRIORITY,20713
LOW_CTR,6869
STALE_CONTENT,236
LOW_CTR_AND_STALE,13


In [63]:
queue["action"].value_counts()

,count
action,
MONITOR,20713
REVIEW_CTR,6869
REVIEW_REFRESH,236
REVIEW_REFRESH_AND_CTR,13


In [64]:
queue.groupby(["reason_code", "action"]).size().reset_index(name="n")

,reason_code,action,n
0,LOW_CTR,REVIEW_CTR,6869
1,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR,13
2,LOW_PRIORITY,MONITOR,20713
3,STALE_CONTENT,REVIEW_REFRESH,236


In [65]:
import os

output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "baseline_action_score.csv"
)

queue.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(queue))
print("Columns:", list(queue.columns))

Saved: work/outputs/baseline_action_score.csv
Rows: 27831
Columns: ['client_hash_id', 'content_hash_id', 'total_impressions', 'total_clicks', 'ctr_pct', 'avg_position', 'days_since_update', 'score', 'reason_code', 'action']


In [66]:
import pandas as pd
pd.read_csv(output_path).head(10)

,client_hash_id,content_hash_id,total_impressions,total_clicks,ctr_pct,avg_position,days_since_update,score,reason_code,action
0,client_c182d11e4862a37d,content_42ce26be1ec6be00,4411.0,6.0,0.136,4.26,264,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
1,client_c182d11e4862a37d,content_bea86ce3455100b0,3670.0,1.0,0.027,6.56,232,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
2,client_c182d11e4862a37d,content_5120dcbbb086843d,1429.0,0.0,0.000,6.32,247,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
3,client_c182d11e4862a37d,content_9dc017d4ef83c0d9,231.0,0.0,0.000,6.23,235,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
4,client_c182d11e4862a37d,content_e3faf06779436e84,218.0,0.0,0.000,7.97,235,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
5,client_c182d11e4862a37d,content_7d986479e843e4cd,205.0,0.0,0.000,4.13,246,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
6,client_c182d11e4862a37d,content_d91b572e9926aeea,130.0,0.0,0.000,7.18,235,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
7,client_20259bd6705d81d4,content_1d2dc429b857b82c,6376.0,8.0,0.125,4.74,124,80,LOW_CTR,REVIEW_CTR
8,client_20259bd6705d81d4,content_f8e84719d52ab1fe,3676.0,0.0,0.000,9.83,124,80,LOW_CTR,REVIEW_CTR
9,client_20259bd6705d81d4,content_96eb80b1ad849418,2134.0,4.0,0.187,6.38,124,80,LOW_CTR,REVIEW_CTR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

I reviewed the top 20 ranked content items using the observed CTR, impressions, position, and days since update. The score is a decision-support baseline, not a prediction of future performance.

| Rank | Action | Reason | Confidence note | What would make it wrong |
|---|---|---|---|---|
| 1 | REVIEW_REFRESH_AND_CTR | Very low CTR with 4,411 impressions and 264 days since update. | High confidence because the item has substantial impressions and is very stale. | It could be wrong if the low CTR is caused by search intent or query mix rather than the content itself. |
| 2 | REVIEW_REFRESH_AND_CTR | Very low CTR with 3,670 impressions and 232 days since update. | High confidence because there is enough observed traffic and strong staleness. | It could be wrong if the page is intentionally targeting low-click informational queries. |
| 3 | REVIEW_REFRESH_AND_CTR | Zero clicks from 1,429 impressions and 247 days since update. | High confidence on the observed signal. | It could be wrong if the impressions come from poorly matched queries that the content was not intended to serve. |
| 4 | REVIEW_REFRESH_AND_CTR | Zero clicks from 231 impressions and 235 days since update. | Moderate confidence because impressions are lower than the first three. | It could be wrong because the sample is relatively small. |
| 5 | REVIEW_REFRESH_AND_CTR | Zero clicks from 218 impressions and 235 days since update. | Moderate confidence because the content is stale but has limited impressions. | It could be wrong because the observed traffic volume is small. |
| 6 | REVIEW_REFRESH_AND_CTR | Zero clicks from 205 impressions and 246 days since update. | Moderate confidence from the combination of zero clicks and high staleness. | It could be wrong if the page has a narrow search audience. |
| 7 | REVIEW_REFRESH_AND_CTR | Zero clicks from 130 impressions and 235 days since update. | Moderate-to-low confidence because impressions are limited. | It could be wrong because 130 impressions may not be enough to judge CTR reliably. |
| 8 | REVIEW_CTR | Low CTR with 6,376 impressions and 124 days since update. | High confidence in the CTR signal because impressions are substantial. | It could be wrong if the queries producing impressions have weak click intent. |
| 9 | REVIEW_CTR | Zero clicks from 3,676 impressions. | High confidence because the item has substantial impressions. | It could be wrong if the impressions are mostly from irrelevant or low-click queries. |
| 10 | REVIEW_CTR | Low CTR with 2,134 impressions and 124 days since update. | High confidence because there is enough observed traffic to inspect the signal. | It could be wrong if the content is serving queries where clicks are not normally expected. |
| 11 | REVIEW_CTR | Zero clicks from 693 impressions. | Moderate confidence because the item has fewer impressions. | It could be wrong because the sample is smaller than the higher-ranked items. |
| 12 | REVIEW_CTR | Zero clicks from 329 impressions. | Moderate confidence. | It could be wrong because the observed impression count is relatively small. |
| 13 | REVIEW_CTR | Zero clicks from 287 impressions. | Moderate confidence. | It could be wrong because the sample is relatively small. |
| 14 | REVIEW_CTR | Zero clicks from 244 impressions. | Moderate confidence. | It could be wrong because the sample is relatively small. |
| 15 | REVIEW_CTR | Zero clicks from 214 impressions. | Moderate-to-low confidence. | It could be wrong because there are relatively few impressions. |
| 16 | REVIEW_CTR | Zero clicks from 172 impressions. | Moderate-to-low confidence. | It could be wrong because the sample is small. |
| 17 | REVIEW_CTR | Zero clicks from 116 impressions. | Low confidence despite the rule flag. | It could be wrong because the impression count is too small to strongly judge CTR. |
| 18 | REVIEW_CTR | 139,417 impressions and 191 clicks produce a low observed CTR. | High confidence because the impression volume is very large. | It could be wrong if the page is intentionally ranking for queries where impressions are more important than clicks. |
| 19 | REVIEW_CTR | 73,503 impressions and 66 clicks produce a low observed CTR. | High confidence because the impression volume is very large. | It could be wrong if query intent or SERP features explain the low click rate. |
| 20 | REVIEW_CTR | Low observed CTR with substantial impressions. | High confidence in the observed CTR signal. | It could be wrong if the low CTR reflects search intent rather than a content problem. |

The review shows that the strongest picks are generally the items with both a clear signal and enough impressions to make that signal meaningful. Lower-impression items should be treated more cautiously.

In [67]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = queue.head(20).copy()

top20

,client_hash_id,content_hash_id,total_impressions,total_clicks,ctr_pct,avg_position,days_since_update,score,reason_code,action
0,client_c182d11e4862a37d,content_42ce26be1ec6be00,4411.0,6.0,0.136,4.26,264,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
1,client_c182d11e4862a37d,content_bea86ce3455100b0,3670.0,1.0,0.027,6.56,232,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
2,client_c182d11e4862a37d,content_5120dcbbb086843d,1429.0,0.0,0.000,6.32,247,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
3,client_c182d11e4862a37d,content_9dc017d4ef83c0d9,231.0,0.0,0.000,6.23,235,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
4,client_c182d11e4862a37d,content_e3faf06779436e84,218.0,0.0,0.000,7.97,235,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
5,client_c182d11e4862a37d,content_7d986479e843e4cd,205.0,0.0,0.000,4.13,246,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
6,client_c182d11e4862a37d,content_d91b572e9926aeea,130.0,0.0,0.000,7.18,235,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
7,client_20259bd6705d81d4,content_1d2dc429b857b82c,6376.0,8.0,0.125,4.74,124,80,LOW_CTR,REVIEW_CTR
8,client_20259bd6705d81d4,content_f8e84719d52ab1fe,3676.0,0.0,0.000,9.83,124,80,LOW_CTR,REVIEW_CTR
9,client_20259bd6705d81d4,content_96eb80b1ad849418,2134.0,4.0,0.187,6.38,124,80,LOW_CTR,REVIEW_CTR


In [68]:
top20[[
    "content_hash_id",
    "total_impressions",
    "total_clicks",
    "ctr_pct",
    "avg_position",
    "days_since_update",
    "score",
    "reason_code",
    "action"
]]

,content_hash_id,total_impressions,total_clicks,ctr_pct,avg_position,days_since_update,score,reason_code,action
0,content_42ce26be1ec6be00,4411.0,6.0,0.136,4.26,264,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
1,content_bea86ce3455100b0,3670.0,1.0,0.027,6.56,232,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
2,content_5120dcbbb086843d,1429.0,0.0,0.000,6.32,247,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
3,content_9dc017d4ef83c0d9,231.0,0.0,0.000,6.23,235,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
4,content_e3faf06779436e84,218.0,0.0,0.000,7.97,235,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
5,content_7d986479e843e4cd,205.0,0.0,0.000,4.13,246,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
6,content_d91b572e9926aeea,130.0,0.0,0.000,7.18,235,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
7,content_1d2dc429b857b82c,6376.0,8.0,0.125,4.74,124,80,LOW_CTR,REVIEW_CTR
8,content_f8e84719d52ab1fe,3676.0,0.0,0.000,9.83,124,80,LOW_CTR,REVIEW_CTR
9,content_96eb80b1ad849418,2134.0,4.0,0.187,6.38,124,80,LOW_CTR,REVIEW_CTR


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

Some of the lower-ranked picks are weak because they have relatively few impressions. In particular, items with around 100–300 impressions and zero clicks can satisfy the LOW_CTR rule but provide less evidence than items with thousands of impressions.

The weakest picks are therefore the low-impression LOW_CTR items. I would treat these as review candidates rather than confirmed content problems.

### Leakage check

The baseline uses observed signals from the available performance and content data, including:

- GSC impressions
- GSC clicks
- observed CTR
- average position
- days since content update

The rule does not use future-window performance or a future outcome label. It also does not use product flags as an input to the score or action.

The score is therefore a hand-written decision-support baseline based on observed signals rather than a learned prediction.

In [69]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 4: weak-pick and leakage checks

# Show the weakest top-20 picks by impression volume
weak_picks = top20[
    ["content_hash_id", "total_impressions", "ctr_pct",
     "days_since_update", "score", "reason_code", "action"]
].sort_values("total_impressions").head(10)

print("Weakest top-20 picks by impression volume:")
display(weak_picks)


# Confirm the baseline output columns
print("\nBaseline output columns:")
print(list(queue.columns))


# Check for fields that would indicate future labels or product flags
leakage_terms = [
    "label",
    "future",
    "product_flag",
    "flag",
    "outcome"
]

possible_leakage_columns = [
    col for col in queue.columns
    if any(term in col.lower() for term in leakage_terms)
]

print("\nPossible leakage columns:")
print(possible_leakage_columns)

assert len(possible_leakage_columns) == 0, (
    f"Potential leakage columns found: {possible_leakage_columns}"
)

print("\nLeakage check: PASS")
print("No future-label or product-flag columns are present in the baseline output.")

# Confirm the actual scoring inputs
scoring_inputs = [
    "total_impressions",
    "total_clicks",
    "ctr_pct",
    "avg_position",
    "days_since_update"
]

print("\nScoring inputs:")
print(scoring_inputs)

print("\nLeakage check: PASS")
print("The baseline score uses observed performance and content freshness signals only.")
print("No future labels, future-window metrics, or product flags are used.")

Weakest top-20 picks by impression volume:


,content_hash_id,total_impressions,ctr_pct,days_since_update,score,reason_code,action
17,content_010a434cb00ece6e,108.0,0.0,113,80,LOW_CTR,REVIEW_CTR
16,content_e2b50fec4f99bda5,116.0,0.0,141,80,LOW_CTR,REVIEW_CTR
6,content_d91b572e9926aeea,130.0,0.0,235,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
15,content_d8be383030412640,172.0,0.0,113,80,LOW_CTR,REVIEW_CTR
5,content_7d986479e843e4cd,205.0,0.0,246,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
14,content_2f20616a0e739edd,214.0,0.0,113,80,LOW_CTR,REVIEW_CTR
4,content_e3faf06779436e84,218.0,0.0,235,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
3,content_9dc017d4ef83c0d9,231.0,0.0,235,90,LOW_CTR_AND_STALE,REVIEW_REFRESH_AND_CTR
13,content_ac31bf4c099cee37,244.0,0.0,113,80,LOW_CTR,REVIEW_CTR
12,content_5053bfde0fc58260,287.0,0.0,113,80,LOW_CTR,REVIEW_CTR



Baseline output columns:
['client_hash_id', 'content_hash_id', 'total_impressions', 'total_clicks', 'ctr_pct', 'avg_position', 'days_since_update', 'score', 'reason_code', 'action']

Possible leakage columns:
[]

Leakage check: PASS
No future-label or product-flag columns are present in the baseline output.

Scoring inputs:
['total_impressions', 'total_clicks', 'ctr_pct', 'avg_position', 'days_since_update']

Leakage check: PASS
The baseline score uses observed performance and content freshness signals only.
No future labels, future-window metrics, or product flags are used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.